AI Citation:

The research questions and visualization ideas that this data cleaning is a part of is created by us (team members). 

AI was used to help with Pandas syntax and data-cleaning difficulties when needed. 

# 11/10-15 Clean and load data for Project 2 (Quarto dashboard)

## 1. Load, clean, and reshape each CSV file (from wide form to long form (1 row per country per year))

### 1.1. Load Data

In [4]:
import pandas as pd
import glob
import os

# Paths
paths = {
    "education": "data/raw/education.csv",
    "gender":    "data/raw/gender.csv",
    "health":    "data/raw/health.csv",
    "environment": "data/raw/environment.csv",
    "economy":   "data/raw/economy-and-growth.csv",
}

# dfs = {name: pd.read_csv(path) for name, path in paths.items()}

### 1.2. Inspect column names

In [ ]:
dfs['economy'].columns

In [ ]:
dfs['education'].columns

In [ ]:
dfs['environment'].columns

In [ ]:
dfs['gender'].columns

## 2. A reusable function to extract only the indicators for the countries of interest
This function below helps us load each World Bank CSV (in wide format; Health, Education, Gender, Economy, Environment), filter to our countries + indicators, melt to long form, and filter to the year window. 

**(Note: Long form is used as a stepping stone for filtering.)**

In [5]:
COUNTRIES = [
    "China", "Japan", "Korea, Rep.", "Mongolia", "Hong Kong SAR, China",
    "Indonesia", "Malaysia", "Philippines", "Singapore", "Thailand",
    "Vietnam", "Cambodia", "Lao PDR", "Myanmar", "Brunei Darussalam"
]

YEAR_COLS = [str(y) for y in range(1960, 2021)]

def extract_long(path, code_to_name, start_year=1980, end_year=2021):
    """
    Load a wide World Bank CSV, filter to our countries + indicators,
    melt to long form, and filter to the year window.

    Parameters
    ----------
    path : str
        Path to the raw wide CSV.
    code_to_name : dict
        Mapping {Indicator Code: short_column_name}.
    """
    df = pd.read_csv(path)

    # 1. Filter countries (huge memory saving)
    df = df[df["Country Name"].isin(COUNTRIES)]

    # 2. Filter to the exact indicator codes we want
    df = df[df["Indicator Code"].isin(code_to_name.keys())]

    # 3. Melt wide → long
    year_cols = [c for c in df.columns if c in YEAR_COLS]
    long = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Code"],
        value_vars=year_cols,
        var_name="Year",
        value_name="value",
    )

    # 4. Type-cast
    long["Year"] = long["Year"].astype(int)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")

    # 5. Trim to the analysis window
    long = long[(long["Year"] >= start_year) & (long["Year"] <= end_year)]

    # 6. Drop missing observations
    long = long.dropna(subset=["value"])

    # 7. Rename indicator codes to short, dashboard-friendly names
    long["indicator"] = long["Indicator Code"].map(code_to_name)

    return long[["Country Name", "Country Code", "Year", "indicator", "value"]]

### 2.1. Apply the extraction to the health file to get life expectancy, infant mortality, and under5 mortality

Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [ ]:
# NOTE - these three health indicators have reliable coverage from 1980-2010. 

# Health
health_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/health-raw.csv",
    code_to_name={
        "SP.DYN.LE00.IN": "life_expectancy",
        "SP.DYN.IMRT.IN": "infant_mortality",
        "SH.DYN.MORT":    "under5_mortality",
    },
)

# view data for an example country
# life_exp = health_long[health_long["indicator"]=="life_expectancy"]
# life_exp[life_exp["Country Name"] == "Vietnam"]
# health_long.head()

### 2.2. Apply extraction to Education file to get primary enrollment %, secondary enrollment, and literacy rate
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [ ]:
# Education

# 11/15 NOTE while data cleaning:
#   These columns have varied coverage. 
#   secondary_enrollment is relatively stable for many countries, but may be underreported for a handful of more-developed countries (Japan, Singapore, etc.) 


edu_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/education-raw.csv",
    code_to_name={
        "SE.PRM.ENRR":    "primary_enrollment",
        "SE.SEC.ENRR":    "secondary_enrollment",   # fairly reliable; missing for some more-developed countries
        "SE.ADT.LITR.ZS": "literacy_rate",
        "SE.SEC.PROG.ZS": "prog_to_secondary"
    },
)

# Inspect output
# secondary_enrollment = edu_long[edu_long["indicator"] == "prog_to_secondary"]
# secondary_enrollment[secondary_enrollment["Country Name"] == "Malaysia"]

### 2.3. Apply Extraction to Env (Sanitation) to get sanitation_basic_pct, water_basic_pct, water_safe_pct 
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [ ]:
env_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/environment-raw.csv",
    code_to_name={
        "SH.STA.BASS.ZS": "sanitation_basic_pct",               # very complete data after 2000; significant (positive) changes for developing countries
        "SH.STA.SMSS.ZS": "sanitation_safely_managed_pct",      # also very complete 2000-2020
        "SH.H2O.BASW.ZS": "water_basic_pct",                    # also very complete 2000-2020
        "SH.STA.ODFC.ZS": "open_defecation_pct",
        # "SH.H2O.SMDW.ZS": "water_safely_managed_pct",           # many missing values (for developing countries)
    }
)

# inspect output
# env[env["Country Code"] == "VNM"]
# env_long[env_long["Country Code"] == "IDN"][env_long["indicator"] == "open_defecation_pct"]

C:\Users\Andre\AppData\Local\Temp\ipykernel_32780\2472394998.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  env_long[env_long["Country Code"] == "IDN"][env_long["indicator"] == "open_defecation_pct"]


,Country Name,Country Code,Year,indicator,value
2413,Indonesia,IDN,2000,open_defecation_pct,32.582502
2473,Indonesia,IDN,2001,open_defecation_pct,31.086988
2533,Indonesia,IDN,2002,open_defecation_pct,29.607921
2593,Indonesia,IDN,2003,open_defecation_pct,28.145715
2653,Indonesia,IDN,2004,open_defecation_pct,26.699935
2713,Indonesia,IDN,2005,open_defecation_pct,25.272040
2773,Indonesia,IDN,2006,open_defecation_pct,23.861290
2833,Indonesia,IDN,2007,open_defecation_pct,22.468255
2893,Indonesia,IDN,2008,open_defecation_pct,21.092524
2953,Indonesia,IDN,2009,open_defecation_pct,19.735049


### 2.4. Apply extraction to Economy to get gdp_per_capita and gdp_per_capita_ppp
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [ ]:
# Economy
econ_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/economy-and-growth-raw.csv",
    code_to_name={
        "NY.GDP.PCAP.KD": "gdp_per_capita_const2010usd",
        "NY.GDP.PCAP.PP.KD": "gdp_per_capita_PPP",
    },
)

# Check output
# econ_long[econ_long["Country Code"] == "SGP"][econ_long["indicator"] == "gdp_per_capita_PPP"]

### 2.5. Apply extraction to Gender to get adolescent_fertility, married_by_18, teenage_mothers_pct

In [ ]:
# Gender (verify codes via the discovery step first)
gender_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/gender-raw.csv",
    code_to_name={
        "SP.ADO.TFRT":          "adolescent_fertility_per_1000_women",    # NOTE: great coverage for countries in 1980-2010
        # "SP.M18.2024.FE.ZS":    "married_by_18_pct",                # Mostly Missing (drop)
        # "SP.MTR.1519.ZS":      "teenage_mothers_pct",               # Mostly Missing (drop)
    },
)

# inspect output
# gender_long[gender_long["Country Code"] == "CHN"][gender_long["indicator"] == "adolescent_fertility_per_1000_women"]

## 3. Build the output by merging the (filtered) long tables by country (wide form; 1 row per country per year; columns are attributes)

### 3.1. Re-pivot the filtered (long) tables to wide form, preparing to join as columns 
(1 column per indicator, 1 row per country per year)

In [161]:
def long_to_wide(long_df):
    return (
        long_df.pivot_table(
            index=["Country Name", "Country Code", "Year"],
            columns="indicator",
            values="value",
            aggfunc="first",   # there should be no duplicates
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

health_wide  = long_to_wide(health_long)
edu_wide     = long_to_wide(edu_long)
gender_wide  = long_to_wide(gender_long)
econ_wide    = long_to_wide(econ_long)
env_wide     = long_to_wide(env_long)

from functools import reduce
panel = reduce(
    lambda left, right: pd.merge(left, right,
                                 on=["Country Name", "Country Code", "Year"],
                                 how="outer"),
    [health_wide, edu_wide, gender_wide, env_wide, econ_wide],
)

### 3.2. Scaffold the full (Country, Year), and left-join each topic (column/metric/indicator) to it

In other words: Make sure every (country, year) combination from 1980-2020 is present (as one row), even if any rows end up being completely blank (will be dropped later). Then, add each country's entries by column (topic). 

In [162]:
scaffold = pd.MultiIndex.from_product(
    [COUNTRIES, range(1980, 2021)],
    names=["Country Name", "Year"],
).to_frame(index=False)

# Add Country Code via a lookup you already have (from any topic frame)
code_lookup = panel[["Country Name", "Country Code"]].drop_duplicates()
scaffold = scaffold.merge(code_lookup, on="Country Name", how="left")
scaffold = scaffold[["Country Name", "Country Code", "Year"]]

from functools import reduce
panel = reduce(
    lambda left, right: left.merge(right, on=["Country Name", "Country Code", "Year"], how="left"),
    [scaffold, health_wide, edu_wide, gender_wide, env_wide, econ_wide],
)

Sanity checks after merge (check columns and rows are present)

In [163]:
# 1. Expected number of rows (≈ 15 countries × 31 years, minus any gaps)
print(panel.shape)

# 2. Confirm only our 15 countries
print(sorted(panel["Country Name"].unique()))

# 3. Confirm year range
print(panel["Year"].min(), panel["Year"].max())

# 4. Coverage per indicator (proportion of country-years with a value)
coverage = panel.drop(columns=["Country Name", "Country Code", "Year"]).notna().mean().sort_values()
print(coverage)

panel[panel["Country Code"] == "MYS"]

(615, 17)
['Brunei Darussalam', 'Cambodia', 'China', 'Hong Kong SAR, China', 'Indonesia', 'Japan', 'Korea, Rep.', 'Lao PDR', 'Malaysia', 'Mongolia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Vietnam']
1980 2020
literacy_rate                          0.128455
prog_to_secondary                      0.339837
sanitation_safely_managed_pct          0.372358
open_defecation_pct                    0.486179
sanitation_basic_pct                   0.489431
water_basic_pct                        0.508943
secondary_enrollment                   0.622764
gdp_per_capita_PPP                     0.749593
primary_enrollment                     0.772358
infant_mortality                       0.910569
under5_mortality                       0.910569
gdp_per_capita_const2010usd            0.962602
adolescent_fertility_per_1000_women    0.975610
life_expectancy                        0.975610
dtype: float64


,Country Name,Country Code,Year,infant_mortality,life_expectancy,under5_mortality,literacy_rate,primary_enrollment,prog_to_secondary,secondary_enrollment,adolescent_fertility_per_1000_women,open_defecation_pct,sanitation_basic_pct,sanitation_safely_managed_pct,water_basic_pct,gdp_per_capita_PPP,gdp_per_capita_const2010usd
246,Malaysia,MYS,1980,25.2,68.117,30.2,69.516327,95.510696,NaN,54.756229,34.5054,NaN,NaN,NaN,NaN,NaN,3317.270514
247,Malaysia,MYS,1981,23.6,68.461,28.2,NaN,95.266273,NaN,57.190170,32.2402,NaN,NaN,NaN,NaN,NaN,3463.233567
248,Malaysia,MYS,1982,22.1,68.788,26.2,NaN,95.640862,NaN,57.329510,29.9750,NaN,NaN,NaN,NaN,NaN,3583.575911
249,Malaysia,MYS,1983,20.7,69.090,24.5,NaN,96.331757,NaN,60.788811,27.9458,NaN,NaN,NaN,NaN,NaN,3718.160749
250,Malaysia,MYS,1984,19.5,69.366,23.0,NaN,95.902061,NaN,62.972439,25.9166,NaN,NaN,NaN,NaN,NaN,3908.417336
251,Malaysia,MYS,1985,18.4,69.621,21.6,NaN,96.106102,NaN,63.181309,23.8874,NaN,NaN,NaN,NaN,NaN,3767.508397
252,Malaysia,MYS,1986,17.4,69.866,20.3,NaN,NaN,NaN,66.251701,21.8582,NaN,NaN,NaN,NaN,NaN,3708.498030
253,Malaysia,MYS,1987,16.5,70.110,19.2,NaN,94.689262,NaN,68.064629,19.8290,NaN,NaN,NaN,NaN,NaN,3788.115993
254,Malaysia,MYS,1988,15.6,70.358,18.2,NaN,94.788544,NaN,66.333549,19.6816,NaN,NaN,NaN,NaN,NaN,4042.128716
255,Malaysia,MYS,1989,14.9,70.612,17.4,NaN,94.607773,NaN,65.392410,19.5342,NaN,NaN,NaN,NaN,NaN,4280.952859


## 4. Final Export
Export the same data to: 1) a csv, and 2) a parquet file (faster loading into Quarto / Jupyter notebook)

In [ ]:
panel.to_csv("C:/dev/fall-2026/DS-2003-Folder/project_2/data/clean/panel_1980_2010.csv", index=False)

# parquet for faster loading
panel.to_parquet("C:/dev/fall-2026/DS-2003-Folder/project_2/data/clean/panel_1980_2010.parquet")